In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
def make_cv_folds(train_pool, dates, n_splits=5):
    for fold, (train_idx, val_idx) in enumerate(TimeSeriesSplit(n_splits=n_splits).split(dates)):
        train_dates = dates[train_idx]# gives the training dates for that specific fold like what the actual dates are
        val_dates = dates[val_idx]# gives the validation dates for that specific fold like what the actual validation dates are
        assert not set(train_dates) & set(val_dates)
        train_mask = train_pool['FlightDate'].isin(train_dates)# finds the rows that correspond to the dates and returns a boolean array
        val_mask = train_pool['FlightDate'].isin(val_dates)# finds the rows that correspond to the dates and returns a boolean array
        train_fold = train_pool[train_mask]#gets the actual flights rows that belong to the training dates
        val_fold = train_pool[val_mask]# gets the actual flight rows that belong to the validation dates
        yield fold, train_fold, val_fold, train_dates, val_dates

In [3]:
df = pd.read_csv('../data/interim/seattle_ontime_clean.csv') #Loading the csv
df.shape
df.columns


Index(['Unnamed: 0', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek',
       'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline',
       'IATA_CODE_Reporting_Airline',
       ...
       'Div4TailNum', 'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID',
       'Div5WheelsOn', 'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff',
       'Div5TailNum', 'Unnamed: 109'],
      dtype='object', length=111)

In [4]:

df['FlightDate'] = pd.to_datetime(df["FlightDate"])
df['FlightDate'].head()

0   2024-08-01
1   2024-08-02
2   2024-08-03
3   2024-08-04
4   2024-08-01
Name: FlightDate, dtype: datetime64[ns]

In [5]:
post_flight = [
    'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups',
    'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'ArrTime', 'ArrDelayMinutes',
    'ArrDel15', 'ArrivalDelayGroups', 'ActualElapsedTime', 'AirTime',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
    'FirstDepTime', 'TotalAddGTime', 'LongestAddGTime',
    'Cancelled', 'CancellationCode', 'Diverted',
    'DivAirportLandings', 'DivReachedDest', 'DivActualElapsedTime', 'DivArrDelay',
    'DivDistance',
    'Div1Airport', 'Div1AirportID', 'Div1AirportSeqID', 'Div1WheelsOn',
    'Div1TotalGTime', 'Div1LongestGTime', 'Div1WheelsOff', 'Div1TailNum',
    'Div2Airport', 'Div2AirportID', 'Div2AirportSeqID', 'Div2WheelsOn',
    'Div2TotalGTime', 'Div2LongestGTime', 'Div2WheelsOff', 'Div2TailNum',
    'Div3Airport', 'Div3AirportID', 'Div3AirportSeqID', 'Div3WheelsOn',
    'Div3TotalGTime', 'Div3LongestGTime', 'Div3WheelsOff', 'Div3TailNum',
    'Div4Airport', 'Div4AirportID', 'Div4AirportSeqID', 'Div4WheelsOn',
    'Div4TotalGTime', 'Div4LongestGTime', 'Div4WheelsOff', 'Div4TailNum',
    'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID', 'Div5WheelsOn',
    'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff', 'Div5TailNum',
]
drop_cols = [
    'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'Tail_Number',
    'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID',
    'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac',
    'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID',
    'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac',
    'DepTimeBlk', 'ArrTimeBlk', 'Flights', 'DistanceGroup','Origin','Unnamed: 0', 'Unnamed: 109'
]


df = df.drop(columns=post_flight + drop_cols)
df.columns

Index(['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
       'IATA_CODE_Reporting_Airline', 'Flight_Number_Reporting_Airline',
       'Dest', 'CRSDepTime', 'CRSArrTime', 'ArrDelay', 'CRSElapsedTime',
       'Distance'],
      dtype='object')

Post-flight columns like DepDelay, TaxiOut and ActualElapsedTime do not exist when someone is booking a flight. Training on them gives a model that scores well and cannot be deployed, so they are dropped here.

The columns in drop_cols are duplicate encodings of information that other kept columns already carry. Origin is dropped for a different reason, which is that every row is SEA after filtering, so the column is constant and there is no pattern in it to find.

The full reasoning is in `README.md`, "Modeling Guardrails".

In [6]:
cutoff = pd.Timestamp('2025-10-01')#this is the cutoff because it correlates exactly with Q4 and also because it leaves plenty of data for training like about 88% for training and 12% for testing so that theres enough data to train on.
df['DepHour'] = df['CRSDepTime'] // 100
train_pool = df[df['FlightDate'] < cutoff] #gets all the rows to train on that are before this particular date and it also happens to be about 88% of the data 
test = df[df['FlightDate'] >= cutoff] # gets all the rows to test on that are after or on this date which also happens to be about 12% of the data

dates = np.sort(train_pool['FlightDate'].unique())# sorts the dates from training pool to unique dates basically gives you all the unique dates
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates):
    print(fold, train_fold.shape[0], val_fold.shape[0], train_dates.min(), train_dates.max(), val_dates.min(), val_dates.max())


print(len(train_pool), train_pool['FlightDate'].nunique(), train_pool['FlightDate'].min(), train_pool['FlightDate'].max())
print(len(test), test['FlightDate'].nunique(), test['FlightDate'].min(), test['FlightDate'].max())
assert train_pool['FlightDate'].max() < test['FlightDate'].min()
assert len(train_pool) + len(test) == len(df)


0 41901 51554 2024-01-01T00:00:00.000000000 2024-04-18T00:00:00.000000000 2024-04-19T00:00:00.000000000 2024-08-02T00:00:00.000000000
1 93455 49684 2024-01-01T00:00:00.000000000 2024-08-02T00:00:00.000000000 2024-08-03T00:00:00.000000000 2024-11-16T00:00:00.000000000
2 143139 41431 2024-01-01T00:00:00.000000000 2024-11-16T00:00:00.000000000 2024-11-17T00:00:00.000000000 2025-03-02T00:00:00.000000000
3 184570 47184 2024-01-01T00:00:00.000000000 2025-03-02T00:00:00.000000000 2025-03-03T00:00:00.000000000 2025-06-16T00:00:00.000000000
4 231754 53895 2024-01-01T00:00:00.000000000 2025-06-16T00:00:00.000000000 2025-06-17T00:00:00.000000000 2025-09-30T00:00:00.000000000
285649 639 2024-01-01 00:00:00 2025-09-30 00:00:00
38841 92 2025-10-01 00:00:00 2025-12-31 00:00:00


The cutoff is 2025-10-01. Everything before it is the train pool, which is 639 dates and 285,649 rows. Everything on or after it is test, which is 92 dates and 38,841 rows, and it is not touched until Issue 10. There is no separate validation split, so every candidate model is scored on these same 5 folds.

TimeSeriesSplit is run over the array of unique dates instead of over rows. Flight rows are not equally spaced, since there are 242 to 555 flights per day, while calendar dates are. gap=0 is the default and it is kept deliberately, because no feature in this notebook is lagged or rolling.

More detail in `README.md`, "Evaluation Strategy".

In [7]:
profile_cols = ['IATA_CODE_Reporting_Airline', 'Flight_Number_Reporting_Airline', 'Dest'] # Grouping by
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates): # Calling the function to give the fold no, the rows of the train fold, the val fold, the dates that the train fold has, the val dates
    profile_median = train_fold.groupby(profile_cols)['ArrDelay'].median() # Grouping the rows in the train fold by the profile_cols and then finding the median for each group
    val_fold = val_fold.merge(profile_median.rename('pred').reset_index(), on=profile_cols, how='left') # This merges the profile_median, it first renames the ArrDelay column to pred to reduce confusion, then resets the index to turn the MultiIndex series into a small dataframe. It merges on the columns in profile_median, and if no match is found for that specific row then pred just returns NaN
    print(fold, val_fold['pred'].isna().sum(), len(val_fold)) # This basically talks about per fold, how many predictions had no match in the group and the len of val_fold is also there to see how much of the total rows does that number make

    # So 32% of the val rows in fold zero have no match at all amongst the groups that have been made, the plan is to not have them rely on per-profile median here because each group might only have 1-2 rows and the median from those rows is not really a good prediction since an outlier in such a small amount of rows can skew the median forward and therefore the predictions will be worse. The threshold is that each profile should have at least 10 rows for it to make a prediction using the per-profile grouping


    

0 16924 51554
1 9959 49684
2 6072 41431
3 11989 47184
4 5033 53895


Fold 0 showed 32% of validation rows with no matching profile in that fold's training rows. The real number is higher than that, because a median is currently computed for every profile that appears even once, so a profile with 1 or 2 rows counts as a match instead of counting as a miss.

The threshold checks two things per row. Whether the profile exists in the training data at all, and whether it has at least 10 rows to compute a median from.

More detail in `README.md`, "Naive Baseline".

n=10 was picked by checking fold 0, the thinnest fold, since that is where the choice matters most. At n=10, 39.9% of individual profiles fall below the threshold but only 5.0% of that fold's training rows sit in them, because thin profiles do not carry much row weight.

n=5 excludes only 1.2% of rows but trusts a median computed from as few as 5 points, which is risky given ArrDelay's right tail. n=15 excludes 7.7% and n=20 excludes 10.2%, so both cost data without buying stability. The same threshold is used at every rung and kept fixed across all 5 folds.

More detail in `README.md`, "Naive Baseline".

In [8]:
# Thresholding approach
mae_scores =[] # list to calculate MAE scores per fold
ladder_mae=[] #this is basically to calculate mae per fold of the rows that rung 1 was not able to predict because of not enough values in the profile or that there was no profile like that.They fell on to rung 2 or rung 3
flat_mae =[] # same as above but what this does is that it doesnt go through rung 2 it just replaces the prediciton for rows that rung 1 didnt predict with rung 3 which is the global median.This list stores the mae per fold in that case in order to compare it to the ladder mae
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates):
    profile_stats = train_fold.groupby(profile_cols)['ArrDelay'].agg(['median', 'count'])
    reliable_profiles = profile_stats[profile_stats['count'] >= 10]['median'] # Gives the reliable profiles, the ones that have at least ten rows, and once they are identified their medians are taken. The code works by first giving a boolean mask for profiles with at least 10 rows, and then another filter on profile_stats keeps only those rows, after which their median is taken and count is discarded to keep only the reliable profiles
    val_fold = val_fold.merge(reliable_profiles.rename('pred').reset_index(), on=profile_cols, how='left') # Just does the merge with the validation fold, same as above
    val_fold['rung'] = np.where(val_fold['pred'].notna(),1,np.nan) # This basically creates a new column called rung to track which pred values are filled by which rung, since this is the first merge all the pred values come from rung 1, and this marks them as so in the rung column
    # print(fold, val_fold['pred'].isna().sum(), len(val_fold))

    # Fallback rate increased as predicted because the thin profiles are now not being considered, and if they are not being considered then there are more NaN's which increased the fallback rate

    coarse_stats = train_fold.groupby(['IATA_CODE_Reporting_Airline', 'DepHour'])['ArrDelay'].agg(['median', 'count'])
    reliable_coarse = coarse_stats[coarse_stats['count'] >= 10]['median'] # 10 is the count here as well, to ensure that only the profiles with at least 10 rows have the median sent as a prediction
    val_fold = val_fold.merge(reliable_coarse.rename('pred_coarse').reset_index(), on=['IATA_CODE_Reporting_Airline', 'DepHour'], how='left')
    unresolved = val_fold['pred'].isna() # Creates a snapshot of the pred column to find out rows which have the value of NaN
    val_fold['pred'] = val_fold['pred'].fillna(val_fold['pred_coarse']) # Replaces the prediction with the coarser prediction when pred is NaN, which means there was no match found in the most specific grouping, and replaces it with the prediction from a less specific grouping
    val_fold.loc[unresolved & val_fold['pred'].notna(),'rung'] = 2 # This line basically, what it does is, it finds the rows that were not resolved by rung 1 and does an elementwise AND with rows that are now resolved by both rung1 and rung2, and due to this elementwise AND, the only rows it shows true on are rows that were unresolved by the first rung which are now resolved by the second rung. The .loc[] then appropriately puts the value in the rung column as 2
    # print(fold, val_fold['pred'].isna().sum(), len(val_fold))

    still_unresolved = val_fold['pred'].isna() # Finds rows that are still unresolved after rung 1 and rung 2
    global_median = train_fold['ArrDelay'].median() # Calculate global median for rung 3
    val_fold['pred'] = val_fold['pred'].fillna(global_median) # Fill the NaN values with the global median, the first two rungs ensure that this is done for the least amount of rows as much as possible
    val_fold.loc[still_unresolved, 'rung'] = 3 # This labels the rows that were still unresolved after rung 1 and rung 2, and assigns them the value of 3.
    # print(fold,val_fold['pred'].isna().sum())
    non_rung_1 = val_fold[val_fold['rung'] != 1] #gets all the rows that were not predicited with rung 1
    fold_ladder_mae = (non_rung_1['ArrDelay'] - non_rung_1['pred']).abs().mean() # computes the mae for rows that were not predicted with rung 1 and rung 2
    ladder_mae.append(fold_ladder_mae)
    
    fold_flat_mae = (non_rung_1['ArrDelay']- global_median).abs().mean() #calculates mae on the rows that were not predicted with rung 1 but basically uses global median as the prediction for those rows and calculates the mae accordingly
    flat_mae.append(fold_flat_mae) 
    fold_mae = (val_fold['ArrDelay'] - val_fold['pred']).abs().mean() # calculating MAE for a single fold 
    mae_scores.append(fold_mae) # add ing the MAE for a particular fold to the mae_scores list
    print(fold, val_fold['rung'].value_counts().sort_index().to_dict()) # Gives the breakdown of how many values have been influenced by which rungs

print(np.mean(mae_scores), np.std(mae_scores)) #calculates mean to get the typical error of this approach and std tells how much those numbers differ amongst seperate folds.

#Mean MAE is 20 minutes for this approach and std is small at 1.2 which means theres not much variation in the performance as seasons/time passes

print(np.mean(ladder_mae),np.std(ladder_mae)) # prints out the mean across all 5 folds and the standard deviation to see performance across all five folds
print(np.mean(flat_mae),np.std(flat_mae))# same as above but in the case of when all the non-rung 1 fall straight to rung 3
print(np.mean(flat_mae) - np.mean(ladder_mae)) #tests to see if rung 2 is actually helping compared to just falling back to rung 3

0 {1.0: 30283, 2.0: 20641, 3.0: 630}
1 {1.0: 39274, 2.0: 10143, 3.0: 267}
2 {1.0: 34109, 2.0: 7308, 3.0: 14}
3 {1.0: 34643, 2.0: 12371, 3.0: 170}
4 {1.0: 39394, 2.0: 14356, 3.0: 145}
19.951910939426405 1.209756490038425
20.42569170229878 1.433145591603604
20.84019793146333 1.3210408121534298
0.41450622916454805


The first rung 2 grouped by carrier and destination and lifted 0.149 minutes over falling straight to the flat global median, which is close to nothing. 11 candidate groupings were then scored on the same 5 folds instead of guessing a replacement, and carrier with departure hour won at 0.415 minutes.

Destination actively hurts. Carrier alone lifts 0.210, which beats carrier and destination at 0.149, so destination is not adding signal, it is splitting groups into smaller and noisier ones. Departure hour carries a real effect that destination does not, since median ArrDelay by scheduled departure hour swings about 9 to 10 minutes across the day.

A ceiling check puts the total headroom available to any median based rung at about 1.70 minutes, so a model that does not beat this baseline by much is not necessarily broken.

More detail in `README.md`, "What Went Wrong".

In [9]:
df['dep_minutes'] = (df['CRSDepTime'] // 100) * 60 + (df['CRSDepTime'] % 100) # adds a new column to the df that computes the scheduled departure time in the form of minutes since midnight.
df['arr_minutes'] = (df['CRSArrTime'] // 100) * 60 + (df['CRSArrTime'] % 100) #same thing as above but for scheduled arrival time
df[['CRSDepTime', 'dep_minutes', 'CRSArrTime', 'arr_minutes']].describe() #the minimum is one here because it literally means 12:01 AM as in 1.000 means that 0001 basically and as an integer thats just 1.0000

df['dep_sin'] = np.sin(2 * np.pi * df['dep_minutes'] / 1440) #calculates the angle and then puts it in the sin function to get the coordinate on the circle this is done so that the gap at midnight between 1439 and 1 is no longer there and is consistent with reality where these are only 2 minutes apart
df['dep_cos'] = np.cos( 2 * np.pi * df['dep_minutes'] / 1440) # same but puts it in a cos function this is done so that well if we only used sin then two distinct angles could have the same coordinates which is why we also use cos.

df['arr_sin'] = np.sin(2 * np.pi * df['arr_minutes'] / 1440) #does the same thing as above but for arrival times
df['arr_cos'] = np.cos(2 * np.pi * df['arr_minutes'] / 1440)

CRSDepTime and CRSArrTime are clock readings in HHMM, not quantities. 10:59 to 11:00 is one minute of real time but a 41 unit jump as an integer. Converting to minutes since midnight makes that same step a clean +1.

That conversion does not fix the day boundary. 23:59 becomes 1439 and 00:01 becomes 1, which is 2 minutes apart in real time and 1438 apart as numbers. Putting the angle through sin and cos places each time on a circle, where those two land 0.0087 apart. Both functions are needed, because sin alone maps 6am and 6pm to the same value.

Month and DayOfWeek wrap around too, but at 12 and 7 levels one-hot handles it, so periodic encoding is only worth it for the 1,440 value time columns.

In [10]:
categorical_cols = ['IATA_CODE_Reporting_Airline', 'Dest', 'Month', 'DayOfWeek']
ohe = OneHotEncoder(drop='first')

drop="first" is used for the linear model. One-hot encoding the 11 carriers gives 11 dummy columns that sum to 1 on every row, and the intercept is a coefficient on a hidden column that is always 1, so the intercept column and the sum of the dummies are identical row for row. Any amount can be subtracted from the intercept and added to every dummy coefficient without changing a single prediction, so there is no unique answer and a number like "carrier AS adds 5 minutes" means nothing.

drop="first" removes one dummy column. That category becomes all zeros and has nothing left to absorb a compensating shift.

In [11]:
open_cols = ['IATA_CODE_Reporting_Airline','Dest'] #columns for which the open encoder will be used
closed_cols = ['Month', 'DayOfWeek'] #columns for which the closed encoder will be used

ohe_open = OneHotEncoder(drop='first', handle_unknown='infrequent_if_exist',min_frequency=4)


In [12]:
ohe_closed = OneHotEncoder(drop='first', categories=[list(range(1, 13)), list(range(1, 8))]) 

Two encoders in the ColumnTransformer, split by whether the category set is closed rather than by cardinality. Cardinality misleads here, because Month has 12 values and fails while carrier has 11 and does not.

Month and DayOfWeek can be written out in advance, so they get an explicit categories= of 1-12 and 1-7 and unknown becomes impossible. This is the column that actually breaks. The folds are temporal, so fold 0 trains on January to April and May through August arrive in its validation set as categories the encoder has never seen, in 3 of 5 folds. No min_frequency value fixes that, because those months are not rare, they are absent.

Dest and carrier cannot be enumerated ahead of time, since an airline can add a route at SEA, so they keep min_frequency=4 and handle_unknown="infrequent_if_exist". min_frequency=2 was tried first and built no bucket at all, because the thinnest Dest in folds 0 and 1 has 3 rows.

More detail in `README.md`, "What Went Wrong".

In [13]:
numeric_cols = ['DayofMonth', 'CRSElapsedTime', 'Distance', 'dep_sin', 'dep_cos', 'arr_sin', 'arr_cos']

Year, Quarter, FlightDate and Flight_Number_Reporting_Airline are all still in df and none of them are handed to the ColumnTransformer. Each is left out for a different reason.

Year takes two values here, 2024 and 2025, and the app predicts 2026 onward, so every prediction it will ever make is for a year the model has never seen. Quarter is a deterministic function of Month, so its dummy columns can be reconstructed exactly from Month's, which is the same collinearity that drop="first" exists to fix. FlightDate never recurs, so there is nothing in it to generalise from, and it stays only because it is the split key and the Issue 3 grouping key. Flight_Number_Reporting_Airline has 2,383 distinct values on the train pool against about 120 columns for all four categoricals that were kept, and it stays as the profile key and as the app's user input in Issue 14.

Keeping a column because it is useful for splitting, grouping or the app is a fine reason to keep a column. It is not a reason to feed it to a model.

More detail in `README.md`, "Modeling Guardrails".

In [14]:
preprocessor = ColumnTransformer([
    ('open', ohe_open, open_cols),
    ('closed', ohe_closed, closed_cols),
    ('num', 'passthrough', numeric_cols),
])

An encoder one-hot encodes every column it is given, so handing it the whole dataframe would treat each distinct Distance value as its own category. Distance has 100 distinct values and CRSElapsedTime has 364, so those two alone would come back as about 460 junk columns. ColumnTransformer routes specific columns to specific transformers. It is built from (name, transformer, columns) triples, and the numerics get the string "passthrough", which copies them through untouched.

What decides where a column goes is not whether it is a number. Month and DayOfWeek are integers and still go to a one-hot encoder. The test is whether the number is a label or a quantity. Distance 2400 really is twice 1200, but Month 12 is not twice Month 6.

The output width is not identical in every fold. Fold 0's 41,901 rows contain 85 destinations and fold 4's 231,754 rows contain 93, because each fold's encoder learns its category list from the rows it is fitted on. If fold 0 knew about all 93 it would mean destinations from 2025 had leaked backwards into a model that is only supposed to know early 2024. This is also why the preprocessor has to sit inside a Pipeline instead of being fitted once on the whole train pool and sliced into folds afterward.

More detail in `README.md`, "Evaluation Strategy".

In [15]:
fold0_train = next(make_cv_folds(train_pool,dates))[1] #gives the training fold for fold 0
ohe_open.fit(fold0_train[open_cols]) #this basically fits the open columns and does the actual encoding
print(ohe_open.infrequent_categories_) # prints out which carriers and which destination are in the infrequent bucket
unseen = ohe_open.transform(pd.DataFrame({'IATA_CODE_Reporting_Airline': ['ZZ'], 'Dest': ['ZZZ']})).toarray() #converts the strings to columns basically like OneHotEncoding would do,this is to check where it goes as in do these strings have the value of is_infrequent = 1 or not.
print(unseen)
print(ohe_open.get_feature_names_out())

[None, array(['HDN'], dtype=object)]
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]
['IATA_CODE_Reporting_Airline_AS' 'IATA_CODE_Reporting_Airline_B6'
 'IATA_CODE_Reporting_Airline_DL' 'IATA_CODE_Reporting_Airline_F9'
 'IATA_CODE_Reporting_Airline_HA' 'IATA_CODE_Reporting_Airline_MQ'
 'IATA_CODE_Reporting_Airline_NK' 'IATA_CODE_Reporting_Airline_OO'
 'IATA_CODE_Reporting_Airline_UA' 'IATA_CODE_Reporting_Airline_WN'
 'Dest_ALW' 'Dest_ANC' 'Dest_ATL' 'Dest_AUS' 'Dest_BLI' 'Dest_BNA'
 'Dest_BOI' 'Dest_BOS' 'Dest_BUR' 'Dest_BWI' 'Dest_BZN' 'Dest_CHS'
 'Dest_CLE' 'Dest_CLT' 'Dest_CMH' 'Dest_CVG' 'Dest_DAL' 'Dest_DCA'
 'Dest_DEN' 'Dest_DFW' 'Dest_DTW' 'Dest_EUG' 'Dest_EWR' 'Dest_FAI'
 'Dest_FAT' 'Dest_FCA' 'Dest_FLL' 'Dest_GEG' 'Dest_GTF' 'Dest_HLN'
 

/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


The encoder was run on fold 0, which is the thinnest fold at 41,901 training rows and the exact case where min_frequency=2 had failed. Folds 2 through 4 each have a destination with exactly 1 row, so any of those would have built a bucket anyway and hidden the problem.

infrequent_categories_ came back as [None, array(['HDN'])]. None for carrier means no bucket was built, which is enough to know that an unseen carrier comes out all zeros before transforming anything at all.

Transforming a row unseen in both columns returned a 94 wide vector with a single 1 at Dest_infrequent_sklearn. Dest worked. Carrier is all zeros, and the carrier feature names list only 10 of the 11 with AA missing, so an unseen carrier encodes identically to American Airlines. No encoder setting fixes this honestly, so it is handled as input validation in Issue 14 instead of being distorted into the encoder here.

More detail in `README.md`, "What Went Wrong".